<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2012%20-%202026-05-08%20-%20Correlaciones%2C%20regresiones%20e%20intro%20a%20scikit-learn/clase_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 12 · Correlaciones, regresiones e introducción a scikit-learn**Fecha:** Viernes 8 de mayo de 2026  **Módulo:** Módulo 3 · Modelado y estadística inferencial  ---

## IntroducciónDel EDA al primer modelo. Vemos correlación de Pearson y Spearman, regresión lineal como modelo base, y entrenamos nuestro primer modelo con scikit-learn.

## 1. Correlación · historia y tipos<img src="https://upload.wikimedia.org/wikipedia/commons/3/34/Correlation_coefficient.png" width="520" alt="Coeficientes de correlación">El concepto de correlación lo formalizó **Karl Pearson** en 1895, extendiendo trabajo previo de **Francis Galton**. Galton estudiaba la herencia de la altura entre padres e hijos y observó el fenómeno que llamó *regresión hacia la media*: hijos de padres muy altos tendían a ser altos, pero menos que sus padres. De ahí el nombre "regresión".### Tipos de correlación- **Pearson (r)** mide asociación **lineal** entre dos variables numéricas. Rango [-1, +1]. Sensible a outliers.- **Spearman (ρ)** mide asociación **monotónica** (crece/decrece, no necesariamente lineal). Trabaja sobre los rangos, no sobre los valores. Robusto a outliers.- **Kendall (τ)** similar a Spearman pero se basa en concordancias. Más robusto con pocas observaciones.**Regla práctica**: empezar con Pearson, si el scatter muestra relación no lineal pasar a Spearman.

## 2. Correlación ≠ causaciónEste es probablemente el error analítico más común. Ejemplos clásicos:- El consumo de helado y las muertes por ahogamiento están correlacionados. Causa común: el verano.- Hay correlación entre el número de películas de Nicolas Cage y los ahogamientos en piscinas (Spurious Correlations, Tyler Vigen).Antes de afirmar causalidad hace falta:1. **Temporalidad**: la causa precede al efecto.2. **Ausencia de confounders**: eliminamos explicaciones alternativas.3. **Mecanismo plausible**: existe una explicación causal creíble.En el seminario no vamos a probar causalidad rigurosamente — eso requiere experimentos o métodos de inferencia causal. Pero hay que comunicar con cuidado: decir "A se asocia con B", no "A causa B".

In [ ]:
import pandas as pdimport numpy as npfrom scipy import statsurl = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"df = pd.read_csv(url).dropna(subset=["Age"])print("Pearson (Age vs Fare):", round(df["Age"].corr(df["Fare"]), 3))print("Spearman (Age vs Fare):", round(df["Age"].corr(df["Fare"], method="spearman"), 3))

## 3. Regresión lineal · el modelo base<img src="https://upload.wikimedia.org/wikipedia/commons/3/3a/Linear_regression.svg" width="420" alt="Regresión lineal">La **regresión lineal simple** ajusta una recta `y = β₀ + β₁·x + ε` para minimizar el error cuadrático. La generalización a muchas variables es la **regresión lineal múltiple**:$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \ldots + \beta_k x_k + \varepsilon$$Los coeficientes `β` se estiman con **mínimos cuadrados ordinarios (OLS)**. Cada coeficiente se interpreta como *"el cambio esperado en y cuando xᵢ aumenta en 1 unidad, manteniendo las demás constantes"*.**Por qué empezar aquí**:- Es **interpretable**: los coeficientes tienen significado directo.- Es **rápida** y se entrena en milisegundos.- Sirve de **baseline** para comparar modelos más complejos.- Se extiende naturalmente a la **regresión logística** para clasificación.

## 4. scikit-learn · la librería estándar<img src="https://scikit-learn.org/stable/_static/scikit-learn-logo-small.png" width="160" alt="scikit-learn">**scikit-learn** es la librería más usada para machine learning clásico en Python. Nació en 2007 como proyecto Google Summer of Code y la empujó INRIA (el instituto francés de investigación en computación).Su API es la razón principal de su éxito: **toda estimador** tiene los mismos métodos:```pythonmodelo.fit(X, y)        # entrenarmodelo.predict(X_new)   # predecirmodelo.score(X, y)      # evaluar```Eso permite intercambiar modelos cambiando una línea, y encadenarlos con `Pipeline` y `GridSearchCV`.

## 5. Train/test split · la regla de oro**Nunca** evaluar un modelo sobre los mismos datos con los que se entrenó. Si lo haces, obtendrás métricas optimistas que no se traducen a producción.La forma mínima: partir el dataset en **train** (70–80%) y **test** (20–30%). El test solo se usa al final para evaluar. `train_test_split` hace esto con `random_state` para reproducibilidad.Para selección de hiperparámetros se necesita una tercera partición o **validación cruzada** (mañana lo vemos).

In [ ]:
from sklearn.linear_model import LinearRegressionfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import r2_score, mean_absolute_errorX = df[["Age", "Pclass", "SibSp", "Parch"]]y = df["Fare"]X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)lr = LinearRegression().fit(X_tr, y_tr)pred = lr.predict(X_te)print("R² test:", round(r2_score(y_te, pred), 3))print("MAE test:", round(mean_absolute_error(y_te, pred), 2))print("\nCoeficientes:")for name, coef in zip(X.columns, lr.coef_):    print(f"  {name}: {coef:+.3f}")print(f"  intercepto: {lr.intercept_:+.3f}")

## 6. Regresión logística · el baseline de clasificaciónCuando el objetivo es **clasificar** (supervivencia sí/no, fraude sí/no), la regresión lineal no sirve porque las predicciones salen del rango [0,1]. La **regresión logística** resuelve esto aplicando la función sigmoidea:$$p(y=1 | x) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \ldots)}}$$Los coeficientes se interpretan como **log-odds**: qué tanto cambia el log de la razón de ocurrencia cuando la variable crece en 1 unidad. No es tan directo como la regresión lineal pero sigue siendo el clasificador **más interpretable**.

In [ ]:
from sklearn.linear_model import LogisticRegressionfrom sklearn.metrics import accuracy_score, classification_reportdf = df.dropna(subset=["Age"]).copy()df["SexN"] = (df["Sex"] == "female").astype(int)X = df[["Age", "Pclass", "SibSp", "Parch", "SexN", "Fare"]]y = df["Survived"]X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)pred = clf.predict(X_te)print("Accuracy:", round(accuracy_score(y_te, pred), 3))print(classification_report(y_te, pred))

## 7. Importancia de variablesLos coeficientes (en valor absoluto) indican cuáles variables pesan más en la predicción. Pero **cuidado**: solo son comparables si las variables están en la misma escala. Si una va en años (0–80) y otra en dólares (0–500), la de dólares tendrá un coeficiente más pequeño simplemente por la escala.Solución: **estandarizar** antes con `StandardScaler`.

In [ ]:
import pandas as pdimport matplotlib.pyplot as pltcoefs = pd.Series(clf.coef_[0], index=X.columns).sort_values()coefs.plot.barh(title="Coeficientes (regresión logística)")plt.axvline(0, color="grey", linestyle="--")plt.tight_layout(); plt.show()

---## 🧑‍🤝‍🧑 Trabajo en grupo sobre el caso- Definan variable objetivo y features de su caso.- Entrenen una regresión lineal o logística baseline.- Reporten R²/accuracy y 2 features más importantes.

## 📦 Entregable del díaNotebook `12_modelo_baseline.ipynb` con métricas.

---## 📚 Lecturas adicionalesPara profundizar después de la clase, en [`Lecturas.md`](./Lecturas.md) encontrarás una curaduría de artículos técnicos y de negocios en español (4 por día), con copia local en la subcarpeta [`lecturas/`](./lecturas/) cuando la fuente lo permite.### 🎬 Video recomendado<iframe width="720" height="405" src="https://www.youtube.com/embed/SZyH6YkQqIk" title="Regresión Lineal con scikit-learn · Curso ML con Python" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>**[Regresión Lineal con scikit-learn · Curso ML con Python](https://www.youtube.com/watch?v=SZyH6YkQqIk)** · AprendeIA con Ligdi Gonzalez_Workflow completo: preparar datos, entrenar modelo, predecir y evaluar._